In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import nivapy3 as nivapy
import pandas as pd
import teotil3 as teo

In [2]:
# Connect to JupyterHub's PostGIS database
eng = nivapy.da.connect_postgis()

# Define datasets of interest
# Year for admin. boundaries
admin_year = 2024
reg_gdf = teo.io.get_regine_geodataframe(eng, admin_year)

# Determine hydrological connectivity
reg_gdf = teo.io.assign_regine_hierarchy(
    reg_gdf,
    regine_col="regine",
    regine_down_col="regine_down",
    order_coastal=False,
    nan_to_vass=True,
    land_to_vass=True,
    add_offshore=True,
)

# Build network graph from adjacency matrix
g = teo.model.build_graph(reg_gdf, id_col="regine", next_down_col="regine_down")

Connection successful.
100.00 % of regines assigned.


In [3]:
reg_id = "002.CAA0"
g2 = nx.dfs_tree(g.reverse(), reg_id).reverse()
reg_list = list(g2.nodes)
reg_list

['002.CAA0',
 '002.CAAA0',
 '002.CAAAZ',
 '002.CAAB',
 '002.CAAC',
 '002.CAB1',
 '002.CAB2',
 '002.CABA',
 '002.CABB',
 '002.CAC',
 '002.CAD1',
 '002.CAD2',
 '002.CAD3',
 '002.CAD4',
 '002.CAD5',
 '002.CAD6',
 '002.CAE1',
 '002.CAE2',
 '002.CAE3',
 '002.CAE4',
 '002.CAE5',
 '002.CAE6']

In [4]:
xl_path = r"/home/jovyan/projects/oslofjord_modelling/oslomod_phase3_teotil/data/wwtp_scenarios_summary.xlsx"
df = pd.read_excel(xl_path)

df = df.query("regine in @reg_list")
df.head()

,scenario,anlegg_nr,kilderefnr,anlegg_name,martini_name,martini_river_or_internal,regine,year,activity,kommune,...,bof5_in_tonnes,bof5_out_tonnes,kof_in_tonnes,kof_out_tonnes,ss_in_tonnes,ss_out_tonnes,totn_in_tonnes,totn_out_tonnes,totp_in_tonnes,totp_out_tonnes
140,Baseline,3205.0215.01,0227AL56,Tuen renseanlegg (Nedlagt),NaN,NaN,002.CAA0,2018,Privat avløpsanlegg,Lillestrøm,...,0.066,0.016,0.131,0.033,0.077,0.009,0.013,0.010,0.002,0.000
151,Baseline,3209.0031.01,0235AL59,Kløfta renseanlegg,NaN,NaN,002.CAB1,2017,Avløpsnett og -rensing,Ullensaker,...,202.768,57.918,470.060,116.634,240.125,18.192,39.346,32.263,5.296,0.290
152,Baseline,3209.0031.01,0235AL59,Kløfta renseanlegg,NaN,NaN,002.CAB1,2018,Avløpsnett og -rensing,Ullensaker,...,200.975,62.484,464.708,118.404,222.825,14.044,38.835,35.105,5.307,0.258
153,Baseline,3209.0031.01,0235AL59,Kløfta renseanlegg,NaN,NaN,002.CAB1,2019,Avløpsnett og -rensing,Ullensaker,...,167.907,50.696,432.683,118.329,246.589,27.349,49.668,43.063,5.697,0.440
161,Baseline,3209.0071.01,0235AL62,Gardermoen sentralrenseanlegg,NaN,NaN,002.CAB2,2017,Avløpsnett og -rensing,Ullensaker,...,1145.423,36.188,2744.606,195.539,891.862,62.430,233.369,55.611,27.717,1.477


In [5]:
par = "totn"
for site_id, site_df in df.groupby(["anlegg_nr"]):
    name = site_df.iloc[0]["anlegg_name"]
    sc_df = site_df.groupby("scenario").sum()[
        [f"{par}_in_tonnes", f"{par}_out_tonnes"]
    ] / 3
    for scen in ["Scenario_A", "Scenario_B"]:
        delta = sc_df.loc['Baseline', f"{par}_out_tonnes"] - sc_df.loc[scen, f"{par}_out_tonnes"]
        print(name, scen, delta, 'tonnes')

Tuen renseanlegg (Nedlagt) Scenario_A 0.0 tonnes
Tuen renseanlegg (Nedlagt) Scenario_B 0.0 tonnes
Kløfta renseanlegg Scenario_A 27.434666666666665 tonnes
Kløfta renseanlegg Scenario_B 27.860999999999997 tonnes
Gardermoen sentralrenseanlegg Scenario_A 5.976333333333322 tonnes
Gardermoen sentralrenseanlegg Scenario_B 8.507666666666665 tonnes
Gjerdrum renseanlegg (Nedlagt) Scenario_A 0.0 tonnes
Gjerdrum renseanlegg (Nedlagt) Scenario_B 3.960666666666666 tonnes
Fagerli renseanlegg Scenario_A 0.0 tonnes
Fagerli renseanlegg Scenario_B 0.0 tonnes


In [6]:
par = "totn"
for scen in ["Scenario_A", "Scenario_B"]:
    sc_df = df.groupby("scenario").sum()[
        [f"{par}_in_tonnes", f"{par}_out_tonnes"]
    ] / 3
    delta = sc_df.loc['Baseline', f"{par}_out_tonnes"] - sc_df.loc[scen, f"{par}_out_tonnes"]
    print(scen, delta, 'tonnes')

Scenario_A 33.411 tonnes
Scenario_B 40.32933333333333 tonnes


In [7]:
sc_df

,totn_in_tonnes,totn_out_tonnes
scenario,,
Baseline,303.645333,102.714333
Scenario_A,303.645333,69.303333
Scenario_B,303.645333,62.385000
